## Hybrid Retriever- Combining Dense And Sparse Retriever

In [11]:
# import langchain
# print(langchain.__version__)

import langchain_classic

print(langchain_classic.__version__)

from langchain_classic.retrievers import EnsembleRetriever

print("Success")

!pip list | findstr langchain

1.0.8
Success
langchain                                1.3.14
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.4.9
langchain-experimental                   0.4.2
langchain-huggingface                    1.2.2
langchain-openai                         1.3.5
langchain-protocol                       0.0.18
langchain-qdrant                         1.1.0
langchain-text-splitters                 1.1.2


In [12]:
from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever



In [13]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1710.94it/s]


In [14]:
### Sparse Retriever(BM25)
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ##top- k documents to retriever

## step 4 : Combine with Ensemble Retriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weight=[0.7,0.3]
)


In [15]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000163842B9C10>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000163842ED310>, k=3)], weights=[0.5, 0.5])

In [16]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


### RAG Pipeline with hybrid retriever

In [18]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.

Context:
{context}

Question:
{question}
""")

In [19]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "openai:gpt-3.5-turbo",
    temperature=0.2
)

In [20]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [25]:
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    RunnableMap(
        {
            "context": lambda x: format_docs(
                hybrid_retriever.invoke(x["question"])
            ),
            "question": lambda x: x["question"],
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [26]:
response = rag_chain.invoke(
    {
        "question": "How can I build an app using LLMs?"
    }
)

print(response)

You can build an app using LLMs by using LangChain, which helps build LLM applications.
